# NB11 — S3.1 grouped CV lambda selection + multi-seed confirmation

This notebook implements the two-stage protocol for the modernized S3.1 scorer.

**Stage A — choose lambda without touching canonical VALID**
- use only `scorer_ready_v2_train`;
- grouped 3-fold CV;
- every positive/paired-negative family stays together;
- all families sharing `source_kit_id` stay in the same fold;
- compare `ranking_weight ∈ {0.10, 0.50, 1.00}`;
- same training seed 42 for every fold;
- choose lambda by mean fold ROC-AUC (exact tie → smaller lambda).

**Stage B — freeze lambda and test robustness**
- train on full canonical TRAIN;
- evaluate canonical VALID with seeds 42/43/44;
- seeds are confirmation only; do **not** choose the prettiest seed;
- report mean ± sample std for AUC/FITB/margins/loss.

Finally, run pure LOO on original outfit size >=4 for the selected-lambda **seed42** checkpoint vs frozen V5. TEST is never loaded.


In [ ]:
from pathlib import Path
import copy
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "exp/s3-1-grouped-cv-lambda-seeds"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.colab import drive
drive.mount("/content/drive")

# Input artifacts may be a shortcut to the team-owned ML_Final folder.
ARTIFACT_ROOT = Path("/content/drive/MyDrive/ML_Final")
os.environ["FASHION_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["FASHION_EMBEDDING_CACHE"] = str(ARTIFACT_ROOT / "fashionclip_item_embeddings.pt")
os.environ["FASHION_EMBEDDING_MANIFEST"] = str(ARTIFACT_ROOT / "embedding_manifest_v1.json")
os.environ["FASHION_CORE7_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "core7_drop_v2")
os.environ["FASHION_SCORER_READY_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "scorer_ready_v2")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)

HEAD = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
print("Branch:", BRANCH)
print("Git HEAD:", HEAD)


In [ ]:
import numpy as np
import torch
import yaml

from src.data.runtime_paths import load_runtime_paths
from src.scorer.checkpoint import build_runtime_provenance, load_checkpoint
from src.scorer.dataset import build_dataset_from_runtime
from src.scorer.model import TypeAwarePairwiseScorer
from src.scorer.train import evaluate_epoch, seed_everything, validate_s3_config
from src.scorer.paired_ranking_experiment import (
    build_paired_train_loader,
    evaluate_pure_loo_4plus,
    fit_paired_ranking_scorer,
)
from src.scorer.paired_ranking_cv import (
    build_fold_datasets,
    build_grouped_family_folds,
    build_nonshuffled_loader,
    select_lambda_by_mean_auc,
    summarize_lambda_cv,
)

CONFIG_PATH = REPO_ROOT / "configs" / "scorer_s3_1_grouped_cv.yaml"
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
validate_s3_config(config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Use a GPU runtime; NB11 performs many FP32 training runs."
print("GPU:", torch.cuda.get_device_name(0))

training = config["training"]
cv_cfg = config["experiment"]["cv"]
seed_cfg = config["experiment"]["seed_confirmation"]
assert training["mixed_precision"] is False
assert training["max_epochs"] == 60
assert training["early_stopping_patience"] == 10
assert training["early_stopping_min_epochs"] == 30
assert cv_cfg["ranking_weights"] == [0.10, 0.50, 1.00]
assert seed_cfg["seeds"] == [42, 43, 44]

paths = load_runtime_paths(repo_root=REPO_ROOT)
provenance = build_runtime_provenance(paths, REPO_ROOT)
assert provenance["git_tree_clean"] is True

# Stage A intentionally loads TRAIN only. Canonical VALID is untouched until lambda is frozen.
train_dataset = build_dataset_from_runtime(paths, "train")
assert len(train_dataset) == 30918
assert len(train_dataset.pair_families) == 15459

folds = build_grouped_family_folds(
    train_dataset,
    n_splits=int(cv_cfg["n_splits"]),
    split_seed=int(cv_cfg["split_seed"]),
)
print("TRAIN-only grouped folds:")
for fold in folds:
    print({k: fold[k] for k in ["fold", "train_family_count", "valid_family_count", "train_group_count", "valid_group_count"]})
print("TEST SPLIT NOT LOADED. CANONICAL VALID NOT LOADED YET.")


In [ ]:
# Sanity-check the current scale-preserving category initialization.
seed_everything(int(cv_cfg["training_seed"]))
probe = TypeAwarePairwiseScorer.from_config(config)
with torch.no_grad():
    norms = probe.category_embedding.weight[1:].norm(dim=1)
print("category init policy:", probe.category_embedding_init_policy)
print("category init std:", probe.category_embedding_init_std)
print("expected 1/sqrt(32):", 32 ** -0.5)
print("mean category norm:", float(norms.mean()))
del probe


## Stage A — grouped 3-fold CV on TRAIN only
This is the expensive part: 3 lambdas × 3 folds = 9 runs. Each run has the same modernized S3.1 training protocol; only lambda and fold data differ. Runs are resumable via a small `metrics.json` completion marker.


In [ ]:
CV_ROOT = Path("/content/drive/MyDrive/s3_1_grouped_cv_lambda_seeds/cv")
CV_ROOT.mkdir(parents=True, exist_ok=True)
cv_rows = []

for ranking_weight in [float(x) for x in cv_cfg["ranking_weights"]]:
    for fold in folds:
        fold_number = int(fold["fold"])
        tag = f"lambda_{ranking_weight:.2f}".replace(".", "p")
        run_dir = CV_ROOT / tag / f"fold_{fold_number}"
        metrics_path = run_dir / "metrics.json"
        run_dir.mkdir(parents=True, exist_ok=True)

        if metrics_path.is_file():
            row = json.loads(metrics_path.read_text(encoding="utf-8"))
            cv_rows.append(row)
            print("RESUME completed:", tag, "fold", fold_number, "AUC", row["valid_roc_auc"])
            continue

        print("\n" + "=" * 88)
        print(f"CV lambda={ranking_weight:.2f} fold={fold_number}/{len(folds)}")
        print("=" * 88)

        fold_train, fold_valid = build_fold_datasets(train_dataset, fold)
        run_config = copy.deepcopy(config)
        run_config["training"]["seed"] = int(cv_cfg["training_seed"])
        run_config["experiment"]["active_phase"] = "grouped_cv"
        run_config["experiment"]["active_fold"] = fold_number
        run_config["experiment"]["active_ranking_weight"] = ranking_weight

        seed_everything(int(run_config["training"]["seed"]))
        model = TypeAwarePairwiseScorer.from_config(run_config).to(device)
        train_loader = build_paired_train_loader(fold_train, run_config, num_workers=0)
        valid_loader = build_nonshuffled_loader(fold_valid, run_config, num_workers=0)

        fit_result = fit_paired_ranking_scorer(
            model,
            train_loader,
            valid_loader,
            config=run_config,
            checkpoint_dir=run_dir,
            provenance=provenance,
            ranking_weight=ranking_weight,
            device=device,
        )

        best_model = TypeAwarePairwiseScorer.from_config(run_config).to(device)
        payload = load_checkpoint(
            run_dir / "best.pt",
            model=best_model,
            map_location=device,
            current_provenance=provenance,
        )
        valid = evaluate_epoch(
            best_model,
            valid_loader,
            criterion=torch.nn.BCEWithLogitsLoss(),
            device=device,
        )
        row = {
            "ranking_weight": ranking_weight,
            "fold": fold_number,
            "best_epoch": int(payload["epoch"]),
            "train_family_count": int(fold["train_family_count"]),
            "valid_family_count": int(fold["valid_family_count"]),
            "valid_roc_auc": float(valid["roc_auc"]),
            "valid_fitb_2way": float(valid["fitb_2way"]),
            "mean_logit_margin": float(valid["mean_logit_margin"]),
            "median_logit_margin": float(valid["median_logit_margin"]),
            "valid_loss": float(valid["loss"]),
            "best_path": str(run_dir / "best.pt"),
        }
        metrics_path.write_text(json.dumps(row, indent=2), encoding="utf-8")
        cv_rows.append(row)
        print("CV RESULT:", json.dumps(row, indent=2))

cv_summary = summarize_lambda_cv(cv_rows)
winner = select_lambda_by_mean_auc(cv_summary)
selected_lambda = float(winner["ranking_weight"])

summary_payload = {
    "protocol": "TRAIN-only source_kit grouped 3-fold CV",
    "rows": cv_rows,
    "summary": cv_summary,
    "winner": winner,
}
(CV_ROOT / "cv_summary.json").write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

print("\nGROUPED CV SUMMARY")
print(f"{'lambda':>8} {'mean_AUC':>10} {'std_AUC':>10} {'mean_FITB':>11} {'std_FITB':>10} {'mean_margin':>12}")
for row in cv_summary:
    print(f"{row['ranking_weight']:>8.2f} {row['mean_roc_auc']:>10.6f} {row['std_roc_auc']:>10.6f} {row['mean_fitb_2way']:>11.6f} {row['std_fitb_2way']:>10.6f} {row['mean_logit_margin']:>12.6f}")
print("\nSELECTED LAMBDA BY MEAN CV ROC-AUC:", selected_lambda)
print("Canonical VALID has still not been used for lambda selection.")


## Stage B — freeze lambda, confirm seeds 42/43/44 on canonical VALID
Now and only now do we load canonical VALID. The selected lambda stays fixed. Seed 43/44 are confirmation seeds; we do not pick whichever seed looks best.


In [ ]:
# Reuse the TRAIN dataset/embedding store already in memory; load canonical VALID now.
valid_dataset = build_dataset_from_runtime(
    paths, "valid", embedding_store=train_dataset.embedding_store
)
assert len(valid_dataset) == 2284
assert len(valid_dataset.pair_families) == 1142

SEED_ROOT = Path("/content/drive/MyDrive/s3_1_grouped_cv_lambda_seeds/seed_confirmation")
SEED_ROOT.mkdir(parents=True, exist_ok=True)
seed_rows = []

for seed in [int(x) for x in seed_cfg["seeds"]]:
    tag = f"lambda_{selected_lambda:.2f}_seed_{seed}".replace(".", "p")
    run_dir = SEED_ROOT / tag
    metrics_path = run_dir / "metrics.json"
    run_dir.mkdir(parents=True, exist_ok=True)

    if metrics_path.is_file():
        row = json.loads(metrics_path.read_text(encoding="utf-8"))
        seed_rows.append(row)
        print("RESUME completed seed", seed, "AUC", row["valid_roc_auc"])
        continue

    print("\n" + "=" * 88)
    print(f"SEED CONFIRMATION lambda={selected_lambda:.2f} seed={seed}")
    print("=" * 88)

    run_config = copy.deepcopy(config)
    run_config["training"]["seed"] = seed
    run_config["experiment"]["active_phase"] = "seed_confirmation"
    run_config["experiment"]["selected_ranking_weight"] = selected_lambda

    seed_everything(seed)
    model = TypeAwarePairwiseScorer.from_config(run_config).to(device)
    train_loader = build_paired_train_loader(train_dataset, run_config, num_workers=0)
    valid_loader = build_nonshuffled_loader(valid_dataset, run_config, num_workers=0)

    fit_paired_ranking_scorer(
        model,
        train_loader,
        valid_loader,
        config=run_config,
        checkpoint_dir=run_dir,
        provenance=provenance,
        ranking_weight=selected_lambda,
        device=device,
    )

    best_model = TypeAwarePairwiseScorer.from_config(run_config).to(device)
    payload = load_checkpoint(
        run_dir / "best.pt",
        model=best_model,
        map_location=device,
        current_provenance=provenance,
    )
    valid_loader = build_nonshuffled_loader(valid_dataset, run_config, num_workers=0)
    valid = evaluate_epoch(
        best_model, valid_loader, criterion=torch.nn.BCEWithLogitsLoss(), device=device
    )
    row = {
        "ranking_weight": selected_lambda,
        "seed": seed,
        "best_epoch": int(payload["epoch"]),
        "valid_roc_auc": float(valid["roc_auc"]),
        "valid_fitb_2way": float(valid["fitb_2way"]),
        "mean_logit_margin": float(valid["mean_logit_margin"]),
        "median_logit_margin": float(valid["median_logit_margin"]),
        "valid_loss": float(valid["loss"]),
        "best_path": str(run_dir / "best.pt"),
    }
    metrics_path.write_text(json.dumps(row, indent=2), encoding="utf-8")
    seed_rows.append(row)
    print("SEED RESULT:", json.dumps(row, indent=2))

def mean_std(values):
    values = np.asarray(values, dtype=float)
    return float(values.mean()), float(values.std(ddof=1)) if len(values) > 1 else 0.0

seed_summary = {"selected_lambda": selected_lambda, "seed_count": len(seed_rows)}
for key in ["valid_roc_auc", "valid_fitb_2way", "mean_logit_margin", "median_logit_margin", "valid_loss"]:
    mean, std = mean_std([row[key] for row in seed_rows])
    seed_summary[key + "_mean"] = mean
    seed_summary[key + "_std"] = std

seed_payload = {"rows": seed_rows, "summary": seed_summary}
(SEED_ROOT / "seed_confirmation_summary.json").write_text(json.dumps(seed_payload, indent=2), encoding="utf-8")

print("\nMULTI-SEED CONFIRMATION")
print(f"selected lambda = {selected_lambda:.2f}")
print(f"{'seed':>6} {'epoch':>6} {'AUC':>10} {'FITB':>10} {'mean_margin':>12} {'loss':>10}")
for row in sorted(seed_rows, key=lambda r: r["seed"]):
    print(f"{row['seed']:>6} {row['best_epoch']:>6} {row['valid_roc_auc']:>10.6f} {row['valid_fitb_2way']:>10.6f} {row['mean_logit_margin']:>12.6f} {row['valid_loss']:>10.6f}")
print(f"AUC  mean ± std: {seed_summary['valid_roc_auc_mean']:.6f} ± {seed_summary['valid_roc_auc_std']:.6f}")
print(f"FITB mean ± std: {seed_summary['valid_fitb_2way_mean']:.6f} ± {seed_summary['valid_fitb_2way_std']:.6f}")
print("No seed was selected after looking at these results. Seed42 remains the canonical comparison seed.")
print("TEST SPLIT WAS NOT LOADED.")


## Final downstream check — pure LOO n>=4, seed42 only
This keeps the downstream comparison aligned with the predeclared canonical seed and avoids the 2-item extrapolation for original size-3 outfits.


In [ ]:
seed42_row = next(row for row in seed_rows if int(row["seed"]) == 42)
selected_config = copy.deepcopy(config)
selected_config["training"]["seed"] = 42

candidate_model = TypeAwarePairwiseScorer.from_config(selected_config).to(device)
load_checkpoint(
    seed42_row["best_path"],
    model=candidate_model,
    map_location=device,
    current_provenance=provenance,
)
candidate_model.eval()

canonical_config = yaml.safe_load(
    (REPO_ROOT / "configs" / "scorer_type_aware_pairwise_v1_val_auc.yaml").read_text(encoding="utf-8")
)
frozen_path = REPO_ROOT / "artifacts/checkpoints/type_aware_pairwise_v1/final_val_auc_v5_seed42/best.pt"
baseline_model = TypeAwarePairwiseScorer.from_config(canonical_config).to(device)
load_checkpoint(
    frozen_path,
    model=baseline_model,
    map_location=device,
    current_provenance=provenance,
)
baseline_model.eval()

print("Running pure LOO n>=4 for frozen V5...")
baseline_loo = evaluate_pure_loo_4plus(baseline_model, valid_dataset, device=device)
print("Running pure LOO n>=4 for selected-lambda seed42...")
candidate_loo = evaluate_pure_loo_4plus(candidate_model, valid_dataset, device=device)

loo_summary = {
    "scope": "canonical validation negatives, original outfit size >=4 only",
    "selected_lambda": selected_lambda,
    "seed": 42,
    "baseline_v5": {k: v for k, v in baseline_loo.items() if k != "records"},
    "candidate": {k: v for k, v in candidate_loo.items() if k != "records"},
}
for key in ["top1_localization_accuracy", "hit_at_2", "hit_at_3"]:
    loo_summary.setdefault("delta", {})[key] = float(candidate_loo[key]) - float(baseline_loo[key])

LOO_PATH = Path("/content/drive/MyDrive/s3_1_grouped_cv_lambda_seeds/loo_seed42.json")
LOO_PATH.write_text(json.dumps(loo_summary, indent=2), encoding="utf-8")
print("\nPURE LOO COMPARISON")
print(json.dumps(loo_summary, indent=2))
print("Saved:", LOO_PATH)
print("TEST SPLIT WAS NOT LOADED.")
